# Introducción al Machine Learning a través del Modelado Computacional

**Objetivo:** aprender los fundamentos de programación, matemáticas y modelado usando una simulación concreta como hilo conductor: la trayectoria de una pelota que rebota.

Al finalizar este notebook vas a poder:
- Usar estructuras de control de flujo (`if`, `for`) y funciones en Python
- Trabajar con `numpy` y `matplotlib` para cálculo numérico y visualización
- Entender qué es un modelo, qué son los parámetros y qué significa convergencia
- Conectar estos conceptos con el Machine Learning

> 📖 **Material de lectura complementario:** `notas_intro_ml.md` — definiciones formales, fórmulas y tablas resumen de todos los conceptos de este notebook.

---

## Parte 1 — Fundamentos de Python

Antes de construir la simulación, necesitamos entender los bloques básicos del lenguaje. Los presentamos en el orden en que aparecen dentro de la función principal.

### 1.1 Toma de decisiones: `if` / `else`

El bloque `if` ejecuta código **solo si una condición es verdadera**. Si la condición es falsa, puede ejecutar el bloque `else`.

En la simulación lo usamos para detectar dos eventos:
1. ¿La pelota cruzó el suelo? → aplicar rebote
2. ¿Tiene suficiente velocidad tras el rebote? → contar rebote o detener la pelota

> 📖 Ver en las notas: *Sección 1 — `if` / `else`*

In [ ]:
# Ejemplo: detectar si la pelota tocó el suelo
posicion_y = -0.3  # metros (negativo = pasó del suelo)

if posicion_y <= 0:
    print("La pelota tocó el suelo → aplicar rebote")
else:
    print("La pelota está en el aire → aplicar gravedad")

El `if` puede anidarse para decisiones en cascada. Dentro del rebote, decidimos si la pelota tiene suficiente energía para seguir:

In [ ]:
# if anidado: ¿la pelota rebota o se detiene?
e = 0.75
velocidad_impacto = -0.4          # m/s (negativa: va hacia abajo)

velocidad_rebote = -velocidad_impacto * e   # invertir dirección, aplicar pérdida de energía

if velocidad_rebote > 0.5:
    print(f"Rebote válido: velocidad = {velocidad_rebote:.2f} m/s")
else:
    velocidad_rebote = 0
    print("Energía insuficiente: la pelota se detiene")

### 1.2 Repetición: el bucle `for`

El bucle `for` recorre **cada elemento de una secuencia** y ejecuta el mismo bloque de código para cada uno. En la simulación lo usamos de dos formas:
- Para avanzar el tiempo paso a paso (la trayectoria de una pelota)
- Para evaluar miles de candidatos en la búsqueda de parámetros

> 📖 Ver en las notas: *Sección 1 — `for`*

In [ ]:
# Simular 5 pasos de tiempo: la gravedad acelera la pelota hacia abajo
g  = 9.81   # m/s²
dt = 0.1    # segundos por paso
v  = 0.0    # velocidad inicial (en reposo)

for paso in range(5):
    v = v - g * dt      # la gravedad suma velocidad negativa (hacia abajo)
    print(f"Paso {paso + 1}: velocidad = {v:.2f} m/s")

### 1.3 Listas: almacenar el historial

Una **lista** es una colección ordenada y mutable de elementos. La usamos para guardar el registro de posiciones a lo largo del tiempo: en cada paso guardamos una tupla `(tiempo, altura)`.

> 📖 Ver en las notas: *Sección 1 — Listas*

In [ ]:
# Registrar la trayectoria como lista de tuplas (tiempo, altura)
trayectoria = []    # lista vacía al inicio

y, v, t = 10.0, 0.0, 0.0

for _ in range(6):
    trayectoria.append((round(t, 2), round(y, 3)))   # guardar estado actual
    v = v - 9.81 * 0.1
    y = y + v * 0.1
    t += 0.1

print("Historial (tiempo, altura):")
for registro in trayectoria:
    print(f"  t={registro[0]}s  →  y={registro[1]}m")

### 1.4 Funciones: encapsular lógica reutilizable

Una función (`def`) agrupa código bajo un nombre, acepta parámetros y retorna resultados. Permite ejecutar la misma lógica con distintos valores sin repetir código.

En la simulación, encapsulamos toda la lógica en `simular_rebotes(e)`. Esto nos permite llamarla miles de veces con distintos valores de `e` sin reescribir nada.

> 📖 Ver en las notas: *Sección 1 — Funciones (`def`)*

In [ ]:
# Función mínima: un paso de integración de Euler
def paso_euler(y, v, g=9.81, dt=0.1):
    """Avanza posición y velocidad un paso dt bajo gravedad g."""
    v_nuevo = v - g * dt
    y_nuevo = y + v_nuevo * dt
    return y_nuevo, v_nuevo   # retorna dos valores como tupla

# Llamar la función tres veces
y, v = 10.0, 0.0
for i in range(3):
    y, v = paso_euler(y, v)
    print(f"Paso {i+1}: y={y:.3f}m, v={v:.3f}m/s")

---
## Parte 2 — Computación Científica

Para trabajar con miles de datos a la vez necesitamos herramientas más potentes que las listas nativas de Python.

### 2.1 NumPy: arreglos y operaciones vectorizadas

NumPy provee el `ndarray`: un arreglo de datos del mismo tipo, optimizado para operaciones matemáticas. La diferencia clave con una lista es que **una operación se aplica a todos los elementos a la vez**, sin necesitar un `for`.

En ML, los datos, los pesos del modelo y las predicciones se representan siempre como arrays de NumPy.

> 📖 Ver en las notas: *Sección 2 — NumPy y Vectores*

In [ ]:
import numpy as np

# Lista Python vs. Array NumPy: misma operación, distinta sintaxis
lista = [0.4, 0.5, 0.6, 0.7, 0.8]
array = np.array([0.4, 0.5, 0.6, 0.7, 0.8])

# Lista: necesito un for para operar sobre cada elemento
lista_doble = [x * 2 for x in lista]

# Array: la operación se aplica a todos los elementos directamente
array_doble = array * 2

print("Lista  * 2:", lista_doble)
print("Array  * 2:", array_doble)

NumPy también permite filtrar con una máscara booleana, sin necesidad de un `for` con `if`:

In [ ]:
np.random.seed(42)

# Generar 2000 candidatos de e en el rango [0.4, 0.95] de una vez
e_candidatos = np.random.uniform(0.4, 0.95, size=2000)

# Supongamos que ya tenemos los rebotes de cada candidato
rebotes_ejemplo = np.random.randint(3, 12, size=2000)

# Filtrar vectorialmente: sin for, sin append
condicion   = (rebotes_ejemplo >= 5) & (rebotes_ejemplo <= 7)
e_aceptados = e_candidatos[condicion]

print(f"Candidatos totales:   {len(e_candidatos)}")
print(f"Candidatos aceptados: {len(e_aceptados)}")
print(f"Media de aceptados:   {np.mean(e_aceptados):.4f}")

### 2.2 Matplotlib: visualización

Matplotlib crea gráficos estáticos, animados e interactivos. En ML se usa principalmente para explorar datos y monitorear la curva de pérdida durante el entrenamiento.

> 📖 Ver en las notas: *Sección 2 — Matplotlib*

In [ ]:
import matplotlib.pyplot as plt

# Simular una trayectoria corta para visualizarla
tiempos, alturas = [], []
y, v, t = 10.0, 0.0, 0.0

while y >= 0 and t < 2.0:
    tiempos.append(t)
    alturas.append(y)
    v = v - 9.81 * 0.05
    y = y + v * 0.05
    t += 0.05

plt.figure(figsize=(9, 4))
plt.plot(tiempos, alturas, color='#E67E22', linewidth=2, label='Trayectoria')
plt.axhline(0, color='#333333', linewidth=1.5, linestyle='--', label='Suelo')
plt.title('Trayectoria de la pelota (caída libre, sin rebote aún)')
plt.xlabel('Tiempo (s)')
plt.ylabel('Altura (m)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Parte 3 — Fundamentos Matemáticos

Dos conceptos matemáticos son centrales tanto en la simulación como en Machine Learning: las **derivadas** y la **estadística descriptiva**.

### 3.1 Derivadas e integración de Euler

La **derivada** mide la tasa de cambio instantánea de una función. En física:

$$v = \frac{dy}{dt} \qquad a = \frac{dv}{dt} = -g$$

Como la computadora no puede calcular derivadas continuas, las aproximamos con pasos discretos — esto es la **integración de Euler**:

$$v_{t+\Delta t} = v_t - g \cdot \Delta t \qquad y_{t+\Delta t} = y_t + v_{t+\Delta t} \cdot \Delta t$$

**Conexión con ML:** el Gradiente Descendente aplica la misma lógica. En lugar de la derivada de la posición, calcula la derivada de la función de error respecto a los parámetros:

$$\theta_{t+1} = \theta_t - \alpha \cdot \frac{\partial L}{\partial \theta}$$

donde $\alpha$ (tasa de aprendizaje) juega el mismo rol que $\Delta t$.

> 📖 Ver en las notas: *Sección 3 — Derivadas*

In [ ]:
# ¿Cómo afecta el tamaño del paso dt a la precisión?
# Solución exacta para caída libre desde y0=10m: y(t) = y0 - 0.5*g*t²

y0, g, t_final = 10.0, 9.81, 1.0
y_exacto = y0 - 0.5 * g * t_final**2
print(f"Solución exacta a t=1s: y = {y_exacto:.4f} m")
print()

for dt in [0.5, 0.1, 0.01]:
    y, v, t = y0, 0.0, 0.0
    while t < t_final - dt / 2:
        v = v - g * dt
        y = y + v * dt
        t += dt
    error = abs(y - y_exacto)
    print(f"  dt={dt:.2f}s  →  y_euler={y:.4f}m  |  error={error:.4f}m")

### 3.2 Estadística descriptiva: media y varianza

La **media** ($\mu$) describe el valor central de un conjunto de datos:
$$\mu = \frac{1}{n} \sum_{i=1}^{n} x_i$$

La **varianza** ($\sigma^2$) mide cuánto se dispersan los datos alrededor de la media:
$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \mu)^2$$

En la simulación, la media de los $e$ aceptados es nuestra estimación del parámetro correcto, y la varianza mide la incertidumbre sobre esa estimación.

**Conexión con ML:** varianza alta en las predicciones puede indicar *overfitting*.

> 📖 Ver en las notas: *Sección 3 — Estadística Descriptiva*

In [ ]:
# Calcular media y varianza descomponiendo cada paso
datos = np.array([0.72, 0.74, 0.71, 0.75, 0.73, 0.70, 0.76, 0.72])

media        = np.mean(datos)
desviaciones = datos - media          # distancia de cada punto a la media
varianza     = np.mean(desviaciones**2)   # promedio de los cuadrados

print(f"Datos:        {datos}")
print(f"Media:        {media:.4f}")
print(f"Desviaciones: {desviaciones.round(4)}")
print(f"Varianza:     {varianza:.6f}")
print()
print(f"np.mean: {np.mean(datos):.4f}  (idéntico)")
print(f"np.var:  {np.var(datos):.6f}  (idéntico)")

---
## Parte 4 — La Simulación Completa

Ahora integramos todo lo anterior. Construimos el modelo pieza por pieza y lo usamos para buscar el parámetro correcto.

### 4.1 El modelo

Un **modelo** es una representación matemática de un fenómeno real. Toma variables de entrada (inputs), aplica reglas matemáticas, y produce salidas (outputs).

Nuestro modelo:
- **Input:** coeficiente de restitución `e` (el parámetro a encontrar)
- **Reglas:** integración de Euler + detección de colisión con `if`
- **Output:** número de rebotes y trayectoria completa

El **coeficiente de restitución** `e` (entre 0 y 1) define qué fracción de la velocidad se conserva en cada rebote. `e = 1` es perfectamente elástico; `e = 0` significa que la pelota no rebota.

> 📖 Ver en las notas: *Sección 4 — Modelo y Parámetros*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def simular_rebotes(e, g=9.81, y0=10.0, dt=0.01, t_max=100.0):
    """
    Simula la trayectoria de una pelota usando integración de Euler.

    Parámetros
    ----------
    e     : coeficiente de restitución (0 <= e <= 1)
    g     : aceleración gravitacional (m/s²)
    y0    : altura inicial (m)
    dt    : paso de tiempo (s)
    t_max : tiempo máximo de simulación (s)

    Retorna
    -------
    rebotes    : int  — número de rebotes válidos
    trayectoria: list — historial de (tiempo, altura)
    """
    y, v, t = y0, 0.0, 0.0
    rebotes = 0
    trayectoria = [(t, y)]

    while t < t_max:
        # Integración de Euler: avanzar un paso de tiempo
        v_siguiente = v - g * dt
        y_siguiente = y + v_siguiente * dt

        # if: ¿la pelota cruzó el suelo?
        if y_siguiente <= 0:
            y_siguiente = 0
            v_siguiente = -v_siguiente * e   # invertir + aplicar pérdida de energía

            # if anidado: ¿tiene suficiente energía para rebotar?
            if v_siguiente > 0.5:
                rebotes += 1
            else:
                v_siguiente = 0   # detener la pelota

        y, v = y_siguiente, v_siguiente
        t += dt
        trayectoria.append((t, y))

        if y == 0 and v == 0:   # corte temprano
            break

    return rebotes, trayectoria

Antes de lanzar la búsqueda masiva, siempre conviene explorar el modelo manualmente para entender cómo el parámetro afecta el output:

In [ ]:
# Exploración manual: ¿cómo cambia el número de rebotes con e?
print("Coeficiente e  →  Rebotes")
for e_test in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    n, _ = simular_rebotes(e_test)
    barra = '█' * n
    print(f"  e = {e_test}  →  {n:2d} rebotes  {barra}")

### 4.2 Búsqueda de parámetros: muestreo por rechazo

**Problema:** queremos encontrar el valor de `e` que produce entre 5 y 7 rebotes, pero no tenemos una fórmula analítica.

**Estrategia:** generar miles de candidatos aleatorios, simular cada uno, y **rechazar** los que no cumplen la condición. Los que sí la cumplen forman la distribución de parámetros compatibles con el comportamiento deseado.

Esto se llama **muestreo por rechazo** (*rejection sampling*): una forma de inferencia inversa donde, dado un comportamiento observado, encontramos los parámetros que lo producen.

> 📖 Ver en las notas: *Sección 4 — Muestreo por rechazo vs. Monte Carlo*

In [ ]:
np.random.seed(42)
n_iteraciones = 2000
rebotes_min, rebotes_max = 5, 7

# Paso 1: generar candidatos con NumPy (un array, no una lista)
e_candidatos = np.random.uniform(0.4, 0.95, n_iteraciones)

# Paso 2: simular cada candidato y guardar resultados
e_aceptados       = []
todos_los_rebotes = np.zeros(n_iteraciones)

for i, e in enumerate(e_candidatos):
    r, _ = simular_rebotes(e)
    todos_los_rebotes[i] = r

    # Paso 3: aceptar o rechazar según el criterio
    if rebotes_min <= r <= rebotes_max:
        e_aceptados.append(e)

e_aceptados = np.array(e_aceptados)

# Paso 4: estadísticas sobre los aceptados
media_e    = np.mean(e_aceptados)
varianza_e = np.var(e_aceptados)

print(f"Muestras totales:     {n_iteraciones}")
print(f"Muestras aceptadas:   {len(e_aceptados)}")
print(f"Tasa de aceptación:   {len(e_aceptados)/n_iteraciones*100:.1f}%")
print(f"Media  (mu):          {media_e:.4f}")
print(f"Varianza (sigma^2):   {varianza_e:.6f}")

### 4.3 Visualización de resultados

El primer gráfico muestra el *landscape* de la función objetivo: cómo cambia el output del modelo al barrer el espacio de parámetros. La zona azul es la región donde el candidato es aceptado.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(e_candidatos, todos_los_rebotes,
            alpha=0.4, color='#2ECC71', s=12, label='Simulaciones')
plt.axhspan(rebotes_min, rebotes_max,
            color='#4A90E2', alpha=0.2, label='Rango aceptado [5-7 rebotes]')
plt.title('Relacion entre Coeficiente de Restitución (e) y Número de Rebotes')
plt.xlabel('Coeficiente de restitución (e)')
plt.ylabel('Número de rebotes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

El segundo gráfico muestra la distribución de los parámetros aceptados. La media (línea roja) es nuestra estimación más robusta del parámetro correcto.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(e_aceptados, bins=25, color='#4A90E2', edgecolor='white', alpha=0.8)
plt.axvline(media_e, color='#D0021B', linestyle='dashed', linewidth=2,
            label=f'Media ($\mu$) = {media_e:.4f}')
plt.title('Distribución de Parámetros Aceptados')
plt.xlabel('Coeficiente de restitución (e)')
plt.ylabel('Frecuencia')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4.4 Animación de la solución

In [ ]:
rebotes_finales, trayectoria_optima = simular_rebotes(media_e)
tiempos, alturas = zip(*trayectoria_optima)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1)
ax.set_ylim(-0.5, max(alturas) * 1.1)
ax.axis('off')
ax.axhline(0, color='#333333', lw=3)

pelota, = ax.plot([], [], 'o', color='#E67E22', markersize=20)
texto   = ax.text(0.05, 0.95, '', transform=ax.transAxes, fontsize=10,
                  verticalalignment='top', color='#555555')

def inicializar():
    pelota.set_data([], [])
    texto.set_text('')
    return pelota, texto

def animar(i):
    pelota.set_data([0], [alturas[i]])
    texto.set_text(f"t = {tiempos[i]:.1f}s\ne = {media_e:.3f}\nRebotes: {rebotes_finales}")
    return pelota, texto

salto  = max(1, len(alturas) // 150)
frames = range(0, len(alturas), salto)
anim   = FuncAnimation(fig, animar, frames=frames,
                       init_func=inicializar, blit=True, interval=30)
plt.close(fig)
HTML(anim.to_jshtml())

---
## Parte 5 — Convergencia Estadística

### ¿Cuántos experimentos son suficientes?

La **convergencia** es el proceso por el cual una medida estadística se estabiliza a medida que aumenta el número de muestras. Con pocas muestras la estimación fluctúa; con muchas, converge a un valor estable.

Vamos a observar cómo la varianza de los parámetros aceptados evoluciona al agregar más simulaciones. Esto es directamente análogo a la **curva de aprendizaje** en ML.

> 📖 Ver en las notas: *Sección 4 — Convergencia*

In [ ]:
np.random.seed(42)
N_max, paso_N = 5000, 50
rango_N = np.arange(paso_N, N_max + 1, paso_N)

# Pre-calcular todos los resultados de una vez
e_full = np.random.uniform(0.4, 0.95, N_max)
r_full = np.array([simular_rebotes(e)[0] for e in e_full])

# Calcular varianza de forma incremental
varianzas, N_validos = [], []

for n in rango_N:
    condicion = (r_full[:n] >= 5) & (r_full[:n] <= 7)
    e_n = e_full[:n][condicion]
    if len(e_n) > 1:
        varianzas.append(np.var(e_n))
        N_validos.append(n)

plt.figure(figsize=(10, 5))
plt.plot(N_validos, varianzas, color='#8E44AD', linewidth=2,
         label='Varianza muestral ($\sigma^2$)')
plt.axhline(varianzas[-1], color='#2C3E50', linestyle='--',
            label=f'Convergencia: {varianzas[-1]:.6f}')
plt.title('Convergencia: Varianza vs. Número de Experimentos')
plt.xlabel('Número de experimentos (N)')
plt.ylabel('Varianza ($\sigma^2$)')
plt.xscale('log')
plt.grid(True, which='both', alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

print("Con pocas muestras la varianza fluctúa. Pasado cierto punto,")
print("agregar más muestras ya no cambia la estimación: eso es convergencia.")

---
## Parte 6 — Para profundizar: rechazo y Monte Carlo

### ¿Qué es la integración Monte Carlo?

La idea central es simple: si querés saber qué fracción de un intervalo cumple una condición, muestreás puntos al azar y contás cuántos la satisfacen.

Esto generaliza a cualquier integral:

$$\int_a^b f(x) \, dx \approx (b - a) \cdot \frac{1}{N} \sum_{i=1}^{N} f(x_i), \qquad x_i \sim \text{Uniforme}(a, b)$$

No necesitás saber nada sobre $f$ — solo evaluarla en puntos aleatorios y promediar.

### El array booleano es la función indicadora

Cuando escribimos:

```python
condicion = (todos_los_rebotes >= 5) & (todos_los_rebotes <= 7)
```

obtenemos un array de `True` / `False` — uno por cada candidato. Este array es exactamente la **función indicadora** $\mathbf{1}_i$: vale 1 si el candidato es aceptado y 0 si no.

NumPy trata `True` como 1 y `False` como 0, así que promediar ese array da directamente la fracción de aceptados:

```python
np.mean(condicion)  # = fracción de True = integral de 1[condicion] / L
```

In [ ]:
# Mostrar la estructura del array booleano
condicion = (todos_los_rebotes >= 5) & (todos_los_rebotes <= 7)

print("Primeras 15 muestras:")
print(f"  e:         {e_candidatos[:15].round(3)}")
print(f"  rebotes:   {todos_los_rebotes[:15].astype(int)}")
print(f"  condicion: {condicion[:15].astype(int)}   ← 1=aceptado, 0=rechazado")
print()
print(f"np.mean(condicion) = {np.mean(condicion):.4f}  →  {np.mean(condicion)*100:.1f}% de los candidatos son aceptados")

### Cómo se reduce al cálculo de e\*

La media de los $e$ aceptados se puede escribir como un cociente de dos promedios:

$$e^* = \frac{\displaystyle\sum_i e_i \cdot \mathbf{1}_i}{\displaystyle\sum_i \mathbf{1}_i} = \frac{\texttt{np.mean(e\_candidatos * condicion)}}{\texttt{np.mean(condicion)}}$$

- **Numerador:** `e_candidatos * condicion` — los rechazados contribuyen $e_i \cdot 0 = 0$; los aceptados contribuyen su valor de $e$.
- **Denominador:** `np.mean(condicion)` — la fracción de aceptados, que normaliza el promedio.

Esto es matemáticamente idéntico a filtrar y promediar — la diferencia es solo cómo se escribe.

In [ ]:
# Verificar que ambas formas dan el mismo número

# Forma 1: rechazo clásico — filtrar y promediar
e_star_rechazo = np.mean(e_candidatos[condicion])

# Forma 2: Monte Carlo — cociente de dos promedios
e_star_mc = np.mean(e_candidatos * condicion) / np.mean(condicion)

print(f"e* (rechazo):      {e_star_rechazo:.6f}")
print(f"e* (Monte Carlo):  {e_star_mc:.6f}")
print()
print("Son el mismo número.")
print("El rechazo filtra primero y luego promedia.")
print("Monte Carlo pondera con 0/1 y divide por la fracción aceptada.")

---
## Resumen: conexión con Machine Learning

| Concepto en la simulación | Equivalente en Machine Learning |
|---|---|
| Función `simular_rebotes(e)` | Modelo (red neuronal, regresión, etc.) |
| Coeficiente de restitución `e` | Parámetros / pesos del modelo |
| Rango objetivo [5-7 rebotes] | Función de pérdida (loss function) |
| Muestreo por rechazo | Optimización por búsqueda aleatoria |
| Integración Monte Carlo | Estimación de expectativas sobre parámetros |
| Media $\mu$ de parámetros aceptados | Estimación robusta del parámetro óptimo |
| Varianza $\sigma^2$ de parámetros aceptados | Incertidumbre / indicador de overfitting |
| Convergencia de la varianza | Convergencia del entrenamiento |
| `dt` en integración de Euler | Tasa de aprendizaje $\alpha$ en Gradiente Descendente |

> 📖 Ver tabla completa en las notas: `notas_intro_ml.md` — *Sección 5*

---

**Próximos pasos:** regresión lineal, Gradiente Descendente, y el ciclo completo de entrenamiento / validación / test.